In [1]:
import pandas as pd
import numpy as np

In [3]:
# NETTOYAGE
# Entrée  : le fichier DVF brut (1 ligne = 1 parcelle/local impliqué
#           dans une vente, PAS 1 ligne = 1 bien vendu).
# Sorties : - dvf_residentiel_clean.csv  -> pour le clustering (Maison / Appartement / Appartement VEFA)
#           - dvf_terrains_locaux_clean.csv -> terrains nus et locaux commerciaux, mis de côté (logique de prix différente)

RAW_PATH = "ValeursFoncieres-2025.txt"

df = pd.read_csv(RAW_PATH, sep="|", dtype=str, encoding="utf-8",)
print("Lignes brutes :", len(df))

# 1. SUPPRESSION DES COLONNES INUTILES
# pour la segmentation marché (localisation / type / prix)
cols_to_drop = [
    "Identifiant de document", "Reference document",
    "1 Articles CGI", "2 Articles CGI", "3 Articles CGI",
    "4 Articles CGI", "5 Articles CGI",
    "No Volume",
    "1er lot", "2eme lot", "3eme lot", "4eme lot", "5eme lot",
    "Surface Carrez du 2eme lot", "Surface Carrez du 3eme lot",
    "Surface Carrez du 4eme lot", "Surface Carrez du 5eme lot",
    "Identifiant local",
    "Prefixe de section",
]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

Lignes brutes : 3714829


In [4]:
# 2. SUPPRESSION DES DOUBLONS EXACTS
# les lignes identiques à cause de la jointure parcelles x lots
before = len(df)
df = df.drop_duplicates()
print(f"Doublons exacts supprimés : {before - len(df)}")
print("Lignes restantes :", len(df))

Doublons exacts supprimés : 454335
Lignes restantes : 3260494


In [5]:
# CONVERSION DES TYPES
def to_float(serie):
    return pd.to_numeric(
        serie.str.replace(" ", "", regex=False).str.replace(",", ".", regex=False),
        errors="coerce",
    )

df["Valeur fonciere"] = to_float(df["Valeur fonciere"])
df["Surface reelle bati"] = to_float(df["Surface reelle bati"])
df["Surface terrain"] = to_float(df["Surface terrain"])
df["Surface Carrez du 1er lot"] = to_float(df["Surface Carrez du 1er lot"])
df["Nombre pieces principales"] = to_float(df["Nombre pieces principales"])
df["Nombre de lots"] = to_float(df["Nombre de lots"])
df["Date mutation"] = pd.to_datetime(df["Date mutation"], format="%d/%m/%Y", errors="coerce")

# 3. RECONSTITUTION D'UNE MUTATION (= UNE VENTE)
# Le fichier DVF liste UNE LIGNE PAR PARCELLE/LOCAL impliqué dans la vente, pas une ligne par bien vendu. Une maison + sa dépendance + 2 parcelles de terrain donnent 4 lignes pour UNE SEULE transaction.
# Du coup on va regrouper : département + commune + no disposition + date + valeur (ces 5 champs sont identiques pour toutes les lignes d'une même mutation).
id_cols = ["Code departement", "Code commune", "No disposition", "Date mutation", "Valeur fonciere"]

df["id_mutation"] = (df[id_cols].fillna("NA").astype(str).apply(lambda row: "_".join(row.values), axis=1))

print("Nombre de mutations distinctes :", df["id_mutation"].nunique())

Nombre de mutations distinctes : 1329258


In [ ]:
# 4. ADRESSE COMPLETE (avant agrégation, car peut varier légèrement entre les lignes d'une même mutation -> on prendra la 1ère adresse non vide au moment du groupby)
def build_adresse(row):
    parts = [row.get("No voie"), row.get("B/T/Q"), row.get("Type de voie"), row.get("Voie")]
    parts = [str(p) for p in parts if pd.notna(p) and str(p).strip() != ""]
    return " ".join(parts).strip()

df["adresse"] = df.apply(build_adresse, axis=1)